# Joshi Part 7: A Simple Monte Carlo Pricer (Rust)

Rust kernel version of `06_joshi_simple_mc.ipynb`.

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep time = { version = "0.3", features = ["macros"] }
:dep rand = "0.8"

## SimpleMC1: Raw Monte Carlo from Scratch

Direct GBM terminal value sampling with Box-Muller transform.

In [ ]:
use std::f64::consts::PI;

let spot = 100.0_f64;
let strike = 100.0;
let rate = 0.05;
let vol = 0.20;
let t = 1.0;
let n_paths = 100_000;

let mut rng = rand::thread_rng();
let mut payoff_sum = 0.0;

for _ in 0..n_paths {
    let u1: f64 = rand::Rng::gen(&mut rng);
    let u2: f64 = rand::Rng::gen(&mut rng);
    let z = (-2.0 * u1.ln()).sqrt() * (2.0 * PI * u2).cos();
    let s_t = spot * ((rate - 0.5 * vol * vol) * t + vol * t.sqrt() * z).exp();
    payoff_sum += (s_t - strike).max(0.0);
}

let mc_raw = (-rate * t).exp() * payoff_sum / n_paths as f64;
println!("SimpleMC1 - Raw MC Call: {:.4}", mc_raw);

## SimpleMC2: RustQuant GBM Paths

In [ ]:
use RustQuant::stochastics::*;

let gbm = GeometricBrownianMotion::new(rate, vol);
let config = StochasticProcessConfig::new(
    spot, 0.0, t, 1, StochasticScheme::EulerMaruyama, n_paths, true, None,
);
let output = gbm.generate(&config);

let payoff_sum: f64 = output.paths.iter()
    .map(|p| (*p.last().unwrap() - strike).max(0.0))
    .sum();
let mc_gbm = (-rate * t).exp() * payoff_sum / n_paths as f64;
println!("SimpleMC2 - GBM MC Call: {:.4}", mc_gbm);

## SimpleMC3: Full MC Engine + Analytic Comparison

In [ ]:
use time::macros::date;
use RustQuant::instruments::*;
use RustQuant::instruments::options::*;

let expiry = date!(2027 - 03 - 22);
let config_full = StochasticProcessConfig::new(
    spot, 0.0, t, 252, StochasticScheme::EulerMaruyama, n_paths, true, None,
);

let vanilla = EuropeanVanillaOption::new(strike, expiry, TypeFlag::Call);
let mc_engine = vanilla.price_monte_carlo(&gbm, &config_full, rate);

let bsm = BlackScholesMertonBuilder::default()
    .underlying_price(spot).strike_price(strike).volatility(vol)
    .risk_free_rate(rate).cost_of_carry(rate)
    .expiration_date(expiry).option_type(TypeFlag::Call)
    .build().unwrap();

println!("SimpleMC3 - Engine MC Call: {:.4}", mc_engine);
println!("Black-Scholes (exact):     {:.4}", bsm.price());
println!("\nAll Greeks:");
println!("  Delta = {:.4}", bsm.delta());
println!("  Gamma = {:.6}", bsm.gamma());
println!("  Vega  = {:.4}", bsm.vega());
println!("  Theta = {:.4}", bsm.theta());
println!("  Rho   = {:.4}", bsm.rho());

## Strike Sensitivity

In [ ]:
println!("{:<10} {:<12} {:<10} {:<10}", "Strike", "Price", "Delta", "Gamma");
println!("{}", "-".repeat(42));
for k in (80..=120).step_by(5) {
    let opt = BlackScholesMertonBuilder::default()
        .underlying_price(spot).strike_price(k as f64).volatility(vol)
        .risk_free_rate(rate).cost_of_carry(rate)
        .expiration_date(expiry).option_type(TypeFlag::Call)
        .build().unwrap();
    println!("{:<10} {:<12.4} {:<10.4} {:<10.6}", k, opt.price(), opt.delta(), opt.gamma());
}

## Implied Volatility Roundtrip

In [ ]:
let price = bsm.price();
let iv = bsm.implied_volatility(price);
println!("Price from vol=0.20: {:.6}", price);
println!("Recovered IV:        {:.6}", iv);